# CityBrain v2 — Enhanced Pipeline
**COMP 9130 Group 5 | Vancouver Pavement Risk Assessment**

Improvements over v1 baseline:
1. **311 filtered to pavement-related complaints only** (Pothole, Street Repair, etc.)
2. **Traffic proximity feature** (distance to nearest traffic count location)
3. **Construction project proximity** (nearby active projects)
4. **Auxiliary regression loss** (PCI score prediction + classification)
5. **Ordinal-aware label encoding** (respects Low < Medium < High ordering)
6. **Focal Loss + Label Smoothing** (best from tuning experiments)
7. **Best hyperparams from grid search** (lr=0.002, dropout=0.4, hidden=128)

---

In [ ]:
# ============================================================
# 0. Setup
# ============================================================
import os, ast, json, warnings, pickle, itertools
import numpy as np
import pandas as pd
import geopandas as gpd
from scipy.spatial import cKDTree
from shapely.geometry import shape, Point
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score, classification_report
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset, Dataset
warnings.filterwarnings('ignore')

from google.colab import drive
drive.mount('/content/drive')
DATA_DIR = '/content/drive/MyDrive/AI-FinalProject/data'

# DATA_DIR = './data'  # local
SNAP_RADIUS_M = 150
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

LABEL_MAP = {'VERY GOOD': 0, 'GOOD': 0, 'FAIR': 1, 'POOR': 2, 'VERY POOR': 2}
# PCI score mapping for auxiliary regression
PCI_SCORE = {'VERY GOOD': 1.0, 'GOOD': 0.75, 'FAIR': 0.5, 'POOR': 0.25, 'VERY POOR': 0.0}

def safe_load(filename, **kwargs):
    path = os.path.join(DATA_DIR, filename) if not os.path.isabs(filename) else filename
    with open(path, 'r', encoding='utf-8', errors='replace') as f:
        first = f.readline()
    sep = ';' if first.count(';') > first.count(',') else ','
    return pd.read_csv(path, sep=sep, on_bad_lines='skip', **kwargs)

Mounted at /content/drive
Device: cpu


## 1. Load & Merge Pavement Data

In [ ]:
enriched_path = os.path.join(DATA_DIR, '/content/drive/MyDrive/AI-FinalProject/data/pavement_enriched.csv')
if os.path.exists(enriched_path):
    df = pd.read_csv(enriched_path)
    print(f'Loaded pavement_enriched.csv: {len(df):,} rows')
else:
    df_local = safe_load('/content/drive/MyDrive/AI-FinalProject/data/pavement_condition.csv'); df_local['road_type'] = 'local'
    df_major = safe_load('/content/drive/MyDrive/AI-FinalProject/data/pavement_condition_major_road_network.csv'); df_major['road_type'] = 'major'
    df = pd.concat([df_local, df_major], ignore_index=True)
    coords = df['geo_point_2d'].str.split(',', expand=True).astype(float)
    df['lat'], df['lon'] = coords[0], coords[1]

df = df[df['PCI Rating'].isin(LABEL_MAP)].copy()
df['risk_label'] = df['PCI Rating'].map(LABEL_MAP)
df['pci_score'] = df['PCI Rating'].map(PCI_SCORE)  # continuous target for regression
df['source'] = (df['road_type'] == 'major').astype(int)
print(f'After label mapping: {len(df):,}')
print(df['risk_label'].value_counts().sort_index())
print(f'\nPCI score distribution:')
print(df['pci_score'].describe())

Loaded pavement_enriched.csv: 13,764 rows
After label mapping: 13,764
risk_label
0    6080
1    3061
2    4623
Name: count, dtype: int64

PCI score distribution:
count    13764.000000
mean         0.541104
std          0.352711
min          0.000000
25%          0.250000
50%          0.500000
75%          0.750000
max          1.000000
Name: pci_score, dtype: float64


## 2. Feature Engineering — Static Features (Enhanced)
New features:
- `dist_to_traffic_count`: distance (m) to nearest traffic count location
- `near_construction`: number of active construction projects within 300m

In [ ]:
pave_coords = np.column_stack([df['lat'].values, df['lon'].values])
LAT_M, LON_M = 111_000, 73_000

# --- streetuse ---
df_st = pd.read_csv(os.path.join(DATA_DIR, '/content/drive/MyDrive/AI-FinalProject/data/public_streets.csv'), quotechar='"', on_bad_lines='skip')
st_geo = df_st['geo_point_2d'].dropna().apply(ast.literal_eval)
st_coords = np.array([[d['lat'], d['lon']] for d in st_geo])
_, idx = cKDTree(st_coords).query(pave_coords)
df['streetuse'] = df_st.loc[st_geo.index, 'streetuse'].values[idx]
print('streetuse:', df['streetuse'].value_counts().to_dict())

# --- ROW width ---
df_row = safe_load('/content/drive/MyDrive/AI-FinalProject/data/right_of_way_widths.csv')
geo_col = [c for c in df_row.columns if 'geo_point' in c.lower()][0]
width_col = [c for c in df_row.columns if 'width' in c.lower()][0]
row_geo = df_row[geo_col].dropna().apply(lambda s: ast.literal_eval(s) if '{' in str(s) else s)
row_coords = np.array([[d['lat'], d['lon']] for d in row_geo if isinstance(d, dict)])
row_widths = pd.to_numeric(df_row.loc[row_geo.index[:len(row_coords)], width_col], errors='coerce').fillna(0).values
_, idx = cKDTree(row_coords).query(pave_coords)
df['ROW_width'] = row_widths[idx]

# --- repair_count ---
df['repair_count'] = 0
repair_path = os.path.join(DATA_DIR, '/content/drive/MyDrive/AI-FinalProject/data/city_project_package_street.csv')
if os.path.exists(repair_path):
    df_repair = safe_load('/content/drive/MyDrive/AI-FinalProject/data/city_project_package_street.csv')
    geo_cols = [c for c in df_repair.columns if 'geo_point' in c.lower()]
    if geo_cols:
        try:
            rg = df_repair[geo_cols[0]].dropna()
            parsed = rg.apply(lambda s: ast.literal_eval(s) if '{' in str(s) else s)
            rc = np.array([[d['lat'], d['lon']] for d in parsed if isinstance(d, dict)])
            counts = cKDTree(rc * [LAT_M, LON_M]).query_ball_point(pave_coords * [LAT_M, LON_M], r=100)
            df['repair_count'] = [len(c) for c in counts]
        except: pass

# --- segment_density ---
tree_all = cKDTree(pave_coords)
neighbours = tree_all.query_ball_point(pave_coords, r=0.003)
df['segment_density'] = [len(n) - 1 for n in neighbours]

# --- elevation & slope ---
if 'elevation_m' not in df.columns:
    df['elevation_m'] = 0.0; df['slope_pct'] = 0.0
df['elevation_m'] = df['elevation_m'].fillna(df['elevation_m'].median())
df['slope_pct'] = df['slope_pct'].fillna(df['slope_pct'].median())

# --- neighbourhood ---
df['neighbourhood'] = df.get('neighbourhood', pd.Series('Unknown', index=df.index)).fillna('Unknown')

# ==========================================
# NEW FEATURE 1: Distance to traffic count location
# ==========================================
df_traffic = safe_load('/content/drive/MyDrive/AI-FinalProject/data/directional_traffic_count_locations.csv')
traf_geo_col = [c for c in df_traffic.columns if 'geo_point' in c.lower()][0]
traf_geo = df_traffic[traf_geo_col].dropna().apply(lambda s: ast.literal_eval(s) if '{' in str(s) else s)
traf_coords = np.array([[d['lat'], d['lon']] for d in traf_geo if isinstance(d, dict)])
traf_tree = cKDTree(traf_coords * [LAT_M, LON_M])
traf_dists, _ = traf_tree.query(pave_coords * [LAT_M, LON_M])
df['dist_to_traffic_count'] = traf_dists  # in meters
print(f'\ndist_to_traffic_count: mean={df["dist_to_traffic_count"].mean():.0f}m, median={df["dist_to_traffic_count"].median():.0f}m')

# Binary: is this a high-traffic area? (within 200m of a traffic count location)
df['near_traffic_counter'] = (df['dist_to_traffic_count'] <= 200).astype(int)
print(f'near_traffic_counter: {df["near_traffic_counter"].sum()} segments within 200m')

# ==========================================
# NEW FEATURE 2: Construction project proximity
# ==========================================
if os.path.exists(repair_path):
    proj_geo = df_repair[geo_cols[0]].dropna().apply(lambda s: ast.literal_eval(s) if '{' in str(s) else s)
    proj_coords = np.array([[d['lat'], d['lon']] for d in proj_geo if isinstance(d, dict)])
    if len(proj_coords) > 0:
        proj_tree = cKDTree(proj_coords * [LAT_M, LON_M])
        proj_near = proj_tree.query_ball_point(pave_coords * [LAT_M, LON_M], r=300)
        df['near_construction'] = [len(p) for p in proj_near]
    else:
        df['near_construction'] = 0
else:
    df['near_construction'] = 0
print(f'near_construction: mean={df["near_construction"].mean():.2f}, max={df["near_construction"].max()}')

print(f'\nStatic features done. Shape: {df.shape}')

streetuse: {'Arterial': 5877, 'Residential': 5180, 'Collector': 2707}

dist_to_traffic_count: mean=1793m, median=1705m
near_traffic_counter: 189 segments within 200m
near_construction: mean=0.13, max=4

Static features done. Shape: (13764, 22)


## 3. Train/Val/Test Split

In [ ]:
y = df['risk_label'].values
y_pci = df['pci_score'].values.astype(np.float32)  # for regression auxiliary
indices = np.arange(len(df))
idx_train, idx_temp = train_test_split(indices, test_size=0.30, random_state=42, stratify=y)
idx_val, idx_test = train_test_split(idx_temp, test_size=0.50, random_state=42, stratify=y[idx_temp])
print(f'Train: {len(idx_train):,}  Val: {len(idx_val):,}  Test: {len(idx_test):,}')
print(f'Train labels: {np.bincount(y[idx_train])}')

Train: 9,634  Val: 2,065  Test: 2,065
Train labels: [4256 2142 3236]


## 4. (Removed) Target-Encoded Features
> `neigh_high_risk_pct` and `neigh_mean_pci` **removed** — they are target leakage.
> Neighbourhood info is captured via one-hot encoding in Tabular-MLP.

In [ ]:
# neigh_high_risk_pct: REMOVED (target leakage)
#   It encodes label distribution per neighbourhood back as input.
#   Model would memorize "40% of this neighbourhood is High" → predict High.
#   neighbourhood one-hot lets the model learn the same pattern honestly.
#
# neigh_mean_pci: REMOVED (same issue — PCI score IS the label)

print('Target encoding skipped — leaky features removed.')
print('Neighbourhood spatial info captured via one-hot in Tabular-MLP.')

Target encoding skipped — leaky features removed.
Neighbourhood spatial info captured via one-hot in Tabular-MLP.


## 5. Temporal Features — FILTERED 311
**Key change:** Only count pavement-related 311 complaints:
- Pothole Case
- Street Repair Case
- Sidewalk Repair Case
- Pavement Marking Maintenance Case
- Street Surface Water Flooding Case
- Street Construction Concern Case

This removes noise from ~150 irrelevant categories (animal complaints, garbage, business licenses, etc.).

In [ ]:
# --- Weather (same as before) ---
df_w = safe_load('weather_vancouver.csv')
date_col = [c for c in df_w.columns if 'date' in c.lower() or 'time' in c.lower()][0]
df_w['date'] = pd.to_datetime(df_w[date_col], errors='coerce')
df_w = df_w.dropna(subset=['date'])
col_map = {}
for c in df_w.columns:
    cl = c.lower().strip()
    if 'max temp' in cl and 'flag' not in cl: col_map[c] = 'max_temp'
    elif 'min temp' in cl and 'flag' not in cl: col_map[c] = 'min_temp'
    elif 'total precip' in cl and 'flag' not in cl: col_map[c] = 'total_precip'
    elif 'total snow' in cl and 'flag' not in cl: col_map[c] = 'total_snow'
df_w = df_w.rename(columns=col_map)
for c in ['max_temp','min_temp','total_precip','total_snow']:
    if c in df_w.columns: df_w[c] = pd.to_numeric(df_w[c], errors='coerce').fillna(0)
df_w['freeze_thaw'] = ((df_w['max_temp'] > 0) & (df_w['min_temp'] < 0)).astype(int)
df_w['extreme'] = (df_w['total_precip'] > df_w['total_precip'].quantile(0.95)).astype(int)
df_w['year'], df_w['month'] = df_w['date'].dt.year, df_w['date'].dt.month
monthly_wx = df_w.groupby(['year','month']).agg(
    avg_max_temp=('max_temp','mean'), avg_min_temp=('min_temp','mean'),
    total_precip_mm=('total_precip','sum'), total_snow_cm=('total_snow','sum'),
    freeze_thaw_days=('freeze_thaw','sum'), extreme_days=('extreme','sum'),
).reset_index()
print(f'Weather: {len(df_w):,} days')

# --- 311: FILTERED to pavement-related only ---
print('\nLoading 311 (pavement-related only)...')
df_311 = safe_load('311_service_requests_2009_2021.csv', low_memory=False)
print(f'  Raw: {len(df_311):,}')

# Dedup
id_cols = [c for c in df_311.columns if 'case' in c.lower() or 'id' in c.lower()]
if id_cols: df_311 = df_311.drop_duplicates(subset=id_cols[0])

# *** KEY CHANGE: Filter to pavement-related types ***
PAVEMENT_TYPES = [
    'Pothole Case',
    'Street Repair Case',
    'Sidewalk Repair Case',
    'Pavement Marking Maintenance Case',
    'Pavement Markings Case',
    'Street Surface Water Flooding Case',
    'Street Construction Concern Case',
    'Street Cleaning and Debris Pick Up Case',
    'Curb Ramp Request Case',
]
type_col = 'Service request type'
df_311 = df_311[df_311[type_col].isin(PAVEMENT_TYPES)]
print(f'  After pavement filter: {len(df_311):,}')
print(f'  Types kept:')
print(df_311[type_col].value_counts().to_string())

# Parse dates
date_cols = [c for c in df_311.columns if 'open' in c.lower() or 'timestamp' in c.lower() or 'created' in c.lower()]
if not date_cols: date_cols = [c for c in df_311.columns if 'date' in c.lower()]
df_311['date'] = pd.to_datetime(df_311[date_cols[0]], errors='coerce', utc=True)

# Parse coordinates
lat_col = [c for c in df_311.columns if c.lower() in ['latitude','lat']]
lon_col = [c for c in df_311.columns if c.lower() in ['longitude','lon']]
if lat_col and lon_col:
    df_311['c_lat'] = pd.to_numeric(df_311[lat_col[0]], errors='coerce')
    df_311['c_lon'] = pd.to_numeric(df_311[lon_col[0]], errors='coerce')
else:
    geo_col = [c for c in df_311.columns if 'geo_local' in c.lower() or 'geo_point' in c.lower()]
    if geo_col:
        raw = df_311[geo_col[0]].dropna()
        if len(raw) > 0 and '{' in str(raw.iloc[0]):
            parsed = raw.apply(lambda s: ast.literal_eval(s) if isinstance(s, str) else s)
            df_311.loc[raw.index, 'c_lat'] = parsed.apply(lambda d: d.get('lat') if isinstance(d, dict) else None)
            df_311.loc[raw.index, 'c_lon'] = parsed.apply(lambda d: d.get('lon') if isinstance(d, dict) else None)

df_311 = df_311.dropna(subset=['date','c_lat','c_lon'])
df_311['year'], df_311['month'] = df_311['date'].dt.year, df_311['date'].dt.month

# Filter to survey years
survey_years = set(df['Year'].unique())
df_311 = df_311[df_311['year'].isin(survey_years)]
print(f'\n  Filtered to survey years {survey_years}: {len(df_311):,}')

# Snap to segments
pave_m = pave_coords * [LAT_M, LON_M]
pave_tree = cKDTree(pave_m)
c311_m = np.column_stack([df_311['c_lat'].values * LAT_M, df_311['c_lon'].values * LON_M])
dists, seg_idx = pave_tree.query(c311_m)
df_311['seg_idx'] = seg_idx
df_311 = df_311[dists <= SNAP_RADIUS_M]
print(f'  Within {SNAP_RADIUS_M}m: {len(df_311):,}')

# NEW: Also compute type-specific counts (pothole vs other)
df_311['is_pothole'] = (df_311[type_col] == 'Pothole Case').astype(int)
df_311['is_street_repair'] = (df_311[type_col] == 'Street Repair Case').astype(int)

complaint_monthly = df_311.groupby(['seg_idx','year','month']).agg(
    cnt=('seg_idx', 'size'),
    pothole_cnt=('is_pothole', 'sum'),
    repair_cnt=('is_street_repair', 'sum'),
).reset_index()
print(f'\n  Monthly complaint groups: {len(complaint_monthly):,}')

Weather: 2,557 days

Loading 311 (pavement-related only)...
  Raw: 2,083,091
  After pavement filter: 135,647
  Types kept:
Service request type
Street Repair Case                         32461
Street Cleaning and Debris Pick Up Case    28919
Pothole Case                               27914
Street Surface Water Flooding Case         22223
Sidewalk Repair Case                       14895
Street Construction Concern Case            5426
Pavement Marking Maintenance Case           1610
Pavement Markings Case                      1312
Curb Ramp Request Case                       887

  Filtered to survey years {np.int64(2020), np.int64(2021)}: 27,187
  Within 150m: 27,024

  Monthly complaint groups: 22,807


## 6. Build Temporal Tensor (Enhanced: 10 channels)
Added 2 new channels:
- `pothole_count`: pothole complaints that month
- `repair_count_monthly`: street repair complaints that month

In [ ]:
# --- Build temporal tensor (N, 12, 10) — 2 extra channels ---
N = len(df)
N_FEAT = 10  # was 8, now 10
X_temporal = np.zeros((N, 12, N_FEAT), dtype=np.float32)
years = df['Year'].values

seg_to_neigh = df['neighbourhood'].values
neigh_month_tot = {}
for _, r in complaint_monthly.iterrows():
    key = (seg_to_neigh[int(r['seg_idx'])], int(r['year']), int(r['month']))
    neigh_month_tot[key] = neigh_month_tot.get(key, 0) + r['cnt']

for i in range(N):
    yr = years[i]
    wx = monthly_wx[monthly_wx['year'] == yr]
    for _, r in wx.iterrows():
        m = int(r['month']) - 1
        if 0 <= m < 12:
            X_temporal[i,m,0] = r['avg_max_temp']
            X_temporal[i,m,1] = r['avg_min_temp']
            X_temporal[i,m,2] = r['total_precip_mm']
            X_temporal[i,m,3] = r['total_snow_cm']
            X_temporal[i,m,4] = r['freeze_thaw_days']
            X_temporal[i,m,7] = r['extreme_days']

    seg_c = complaint_monthly[(complaint_monthly['seg_idx']==i) & (complaint_monthly['year']==yr)]
    neigh = seg_to_neigh[i]
    for _, r in seg_c.iterrows():
        m = int(r['month']) - 1
        if 0 <= m < 12:
            X_temporal[i,m,5] = r['cnt']       # total pavement complaints
            nt = neigh_month_tot.get((neigh, yr, int(r['month'])), 1)
            X_temporal[i,m,6] = min(r['cnt'] / max(nt, 1), 1.0)
            X_temporal[i,m,8] = r['pothole_cnt']     # NEW: pothole count
            X_temporal[i,m,9] = r['repair_cnt']      # NEW: repair count

df['complaint_total'] = X_temporal[:,:,5].sum(axis=1)
df['pothole_total'] = X_temporal[:,:,8].sum(axis=1)

print(f'X_temporal: {X_temporal.shape}')
print(f'Nonzero complaint-months: {(X_temporal[:,:,5]>0).sum():,}')
print(f'complaint_total: mean={df["complaint_total"].mean():.1f}, max={df["complaint_total"].max():.0f}')
print(f'pothole_total: mean={df["pothole_total"].mean():.1f}, max={df["pothole_total"].max():.0f}')

X_temporal: (13764, 12, 10)
Nonzero complaint-months: 11,364
complaint_total: mean=1.0, max=32
pothole_total: mean=0.2, max=13


## 7. Assemble Branch Tensors (Enhanced)
- Road-MLP: 11 dims (+dist_to_traffic_count, +near_construction)
- Tabular-MLP: ~27 dims (neighbourhood one-hot + ROW_width + complaint_total + pothole_total + near_traffic_counter)
- BiLSTM: 10 channels (was 8)

> **Note:** `neigh_high_risk_pct` and `neigh_mean_pci` removed (target leakage).

In [ ]:
# --- Road-MLP: 11 dims (was 9) ---
su_dummies = pd.get_dummies(df['streetuse'], prefix='su')
for c in ['su_Arterial','su_Collector','su_Residential']:
    if c not in su_dummies.columns: su_dummies[c] = 0

X_road_raw = np.column_stack([
    df['length_(m)'].fillna(df['length_(m)'].median()).values,
    su_dummies[['su_Arterial','su_Collector','su_Residential']].values,
    df['elevation_m'].values,
    df['slope_pct'].values,
    df['repair_count'].values,
    df['segment_density'].values,
    df['source'].values,
    df['dist_to_traffic_count'].values,   # NEW
    df['near_construction'].values,        # NEW
]).astype(np.float32)
print(f'Road-MLP: {X_road_raw.shape[1]} dims')

# --- Tabular-MLP: ~27 dims (NO leaky features) ---
neigh_dummies = pd.get_dummies(df['neighbourhood'], prefix='n')
X_tab_raw = np.column_stack([
    neigh_dummies.values,
    df['ROW_width'].values,
    df['complaint_total'].values,
    df['pothole_total'].values,            # NEW: pothole-specific
    df['near_traffic_counter'].values,     # NEW: traffic proxy
]).astype(np.float32)
print(f'Tabular-MLP: {X_tab_raw.shape[1]} dims')
print('  (neigh_high_risk_pct and neigh_mean_pci REMOVED — target leakage)')

# --- Normalize ---
sc_road = StandardScaler().fit(X_road_raw[idx_train])
sc_tab = StandardScaler().fit(X_tab_raw[idx_train])
sc_temp = StandardScaler().fit(X_temporal[idx_train].reshape(-1, N_FEAT))

def split_norm(X, sc, idxs, seq=False, nf=None):
    if seq:
        n = len(idxs)
        return sc.transform(X[idxs].reshape(-1, nf)).reshape(n, 12, nf).astype(np.float32)
    return sc.transform(X[idxs]).astype(np.float32)

Xr_tr = split_norm(X_road_raw, sc_road, idx_train)
Xr_v  = split_norm(X_road_raw, sc_road, idx_val)
Xr_te = split_norm(X_road_raw, sc_road, idx_test)
Xt_tr = split_norm(X_temporal, sc_temp, idx_train, True, N_FEAT)
Xt_v  = split_norm(X_temporal, sc_temp, idx_val, True, N_FEAT)
Xt_te = split_norm(X_temporal, sc_temp, idx_test, True, N_FEAT)
Xb_tr = split_norm(X_tab_raw, sc_tab, idx_train)
Xb_v  = split_norm(X_tab_raw, sc_tab, idx_val)
Xb_te = split_norm(X_tab_raw, sc_tab, idx_test)
y_tr, y_v, y_te = y[idx_train], y[idx_val], y[idx_test]
y_pci_tr, y_pci_v, y_pci_te = y_pci[idx_train], y_pci[idx_val], y_pci[idx_test]

print(f'Road: {Xr_tr.shape}, Temporal: {Xt_tr.shape}, Tabular: {Xb_tr.shape}')

Road-MLP: 11 dims
Tabular-MLP: 27 dims
  (neigh_high_risk_pct and neigh_mean_pci REMOVED — target leakage)
Road: (9634, 11), Temporal: (9634, 12, 10), Tabular: (9634, 27)


## 8. Model Definitions (Enhanced)
Changes:
- BiLSTM accepts configurable `feat` (now 10)
- CityBrainFusion has **auxiliary regression head** for PCI score
- FocalLoss with label smoothing
- OrdinalLoss that penalizes predictions farther from true label

In [ ]:
# ============================================================
# Loss Functions
# ============================================================
class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0, label_smoothing=0.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.label_smoothing = label_smoothing
    def forward(self, logits, targets):
        n_classes = logits.size(1)
        if self.label_smoothing > 0:
            with torch.no_grad():
                smooth = torch.full_like(logits, self.label_smoothing / (n_classes - 1))
                smooth.scatter_(1, targets.unsqueeze(1), 1.0 - self.label_smoothing)
            log_p = torch.log_softmax(logits, dim=1)
            loss = -(smooth * log_p).sum(dim=1)
        else:
            loss = nn.functional.cross_entropy(logits, targets, reduction='none')
        p = torch.softmax(logits, dim=1)
        p_t = p.gather(1, targets.unsqueeze(1)).squeeze(1)
        focal_weight = (1 - p_t) ** self.gamma
        loss = focal_weight * loss
        if self.alpha is not None:
            loss = self.alpha[targets] * loss
        return loss.mean()


class OrdinalPenalty(nn.Module):
    """Extra penalty for predictions far from true label (|pred - true| > 1)."""
    def __init__(self, weight=0.5):
        super().__init__()
        self.weight = weight
    def forward(self, logits, targets):
        probs = torch.softmax(logits, dim=1)
        # Expected class = sum(class_idx * prob)
        classes = torch.arange(logits.size(1), device=logits.device, dtype=torch.float32)
        expected = (probs * classes.unsqueeze(0)).sum(dim=1)
        # Squared distance from true label
        penalty = (expected - targets.float()) ** 2
        return self.weight * penalty.mean()


# ============================================================
# Model Definitions
# ============================================================
class RoadMLP(nn.Module):
    def __init__(self, dim, emb=16, hidden=128, drop=0.4):
        super().__init__()
        self.enc = nn.Sequential(
            nn.Linear(dim, hidden), nn.BatchNorm1d(hidden), nn.ReLU(), nn.Dropout(drop),
            nn.Linear(hidden, hidden//2), nn.ReLU(),
            nn.Linear(hidden//2, emb), nn.ReLU())
        self.head = nn.Linear(emb, 3)
    def get_embedding(self, x): return self.enc(x)
    def forward(self, x): return self.head(self.enc(x))


class BiLSTMBranch(nn.Module):
    def __init__(self, feat=10, hidden=64, layers=2, drop=0.4, fc_dim=128):
        super().__init__()
        self.lstm = nn.LSTM(feat, hidden, layers, batch_first=True, bidirectional=True,
                            dropout=drop if layers > 1 else 0)
        self.fc = nn.Sequential(nn.Linear(hidden*2, fc_dim), nn.ReLU(), nn.Dropout(drop))
        self.head = nn.Linear(fc_dim, 3)
        self.fc_dim = fc_dim
    def get_embedding(self, x):
        _, (h, _) = self.lstm(x)
        return self.fc(torch.cat([h[-2], h[-1]], dim=1))
    def forward(self, x): return self.head(self.get_embedding(x))


class TabularMLP(nn.Module):
    def __init__(self, dim, emb=16, hidden=128, drop=0.4):
        super().__init__()
        self.enc = nn.Sequential(
            nn.Linear(dim, hidden), nn.BatchNorm1d(hidden), nn.ReLU(), nn.Dropout(drop),
            nn.Linear(hidden, hidden//2), nn.ReLU(),
            nn.Linear(hidden//2, emb), nn.ReLU())
        self.head = nn.Linear(emb, 3)
    def get_embedding(self, x): return self.enc(x)
    def forward(self, x): return self.head(self.enc(x))


class CityBrainFusion(nn.Module):
    """Late-fusion with auxiliary regression head for PCI score."""
    def __init__(self, road, bilstm, tabular, fuse_dim=160, drop=0.4):
        super().__init__()
        self.road, self.bilstm, self.tabular = road, bilstm, tabular
        self.fusion = nn.Sequential(
            nn.Linear(fuse_dim, 128), nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(drop),
            nn.Linear(128, 64), nn.ReLU(), nn.Dropout(drop))
        self.cls_head = nn.Linear(64, 3)       # classification
        self.reg_head = nn.Linear(64, 1)       # regression (PCI score)

    def _fuse(self, xr, xt, xb):
        return self.fusion(torch.cat([
            self.road.get_embedding(xr),
            self.bilstm.get_embedding(xt),
            self.tabular.get_embedding(xb)], dim=1))

    def forward(self, xr, xt, xb):
        h = self._fuse(xr, xt, xb)
        return self.cls_head(h)

    def forward_with_reg(self, xr, xt, xb):
        h = self._fuse(xr, xt, xb)
        return self.cls_head(h), self.reg_head(h).squeeze(1)

    def forward_ablation(self, xr, xt, xb, disable=None):
        er = self.road.get_embedding(xr) if disable != 'road' else torch.zeros_like(self.road.get_embedding(xr))
        et = self.bilstm.get_embedding(xt) if disable != 'bilstm' else torch.zeros_like(self.bilstm.get_embedding(xt))
        eb = self.tabular.get_embedding(xb) if disable != 'tabular' else torch.zeros_like(self.tabular.get_embedding(xb))
        h = self.fusion(torch.cat([er, et, eb], dim=1))
        return self.cls_head(h)

class FusionDS(Dataset):
    def __init__(self, xr, xt, xb, y_cls, y_reg):
        self.xr, self.xt, self.xb = xr, xt, xb
        self.y_cls, self.y_reg = y_cls, y_reg
    def __len__(self): return len(self.y_cls)
    def __getitem__(self, i):
        return self.xr[i], self.xt[i], self.xb[i], self.y_cls[i], self.y_reg[i]

# Class weights
cc = np.bincount(y_tr)
CW = torch.FloatTensor((1.0/cc) / (1.0/cc).sum() * 3).to(DEVICE)
print(f'Class weights: {CW.cpu().numpy()}')
print(f'Models defined (with auxiliary regression head).')

Class weights: [0.6973287 1.3855419 0.9171294]
Models defined (with auxiliary regression head).


## 9. Training Functions

In [ ]:
# ============================================================
# Training
# ============================================================

def train_branch(model, X_tr, X_v, y_tr_, y_v_, epochs=60, bs=256, lr=2e-3,
                 patience=12, criterion=None, verbose=True):
    model = model.to(DEVICE)
    tr_dl = DataLoader(TensorDataset(torch.tensor(X_tr), torch.tensor(y_tr_).long()), batch_size=bs, shuffle=True)
    v_dl  = DataLoader(TensorDataset(torch.tensor(X_v), torch.tensor(y_v_).long()), batch_size=bs)
    if criterion is None:
        criterion = FocalLoss(alpha=CW, gamma=2.0, label_smoothing=0.1)
    ordinal = OrdinalPenalty(weight=0.3)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    best_f1, wait, best_sd = 0, 0, None
    for ep in range(1, epochs+1):
        model.train()
        for X, yy in tr_dl:
            X, yy = X.to(DEVICE), yy.to(DEVICE)
            logits = model(X)
            loss = criterion(logits, yy) + ordinal(logits, yy)
            opt.zero_grad(); loss.backward(); opt.step()
        sch.step()
        model.eval()
        preds, labs = [], []
        with torch.no_grad():
            for X, yy in v_dl:
                preds.extend(model(X.to(DEVICE)).argmax(1).cpu().numpy())
                labs.extend(yy.numpy())
        f1 = f1_score(labs, preds, average='macro')
        if f1 > best_f1: best_f1, wait, best_sd = f1, 0, {k:v.cpu().clone() for k,v in model.state_dict().items()}
        else: wait += 1
        if verbose and ep % 10 == 0: print(f'  Ep {ep:>3} val_f1={f1:.4f} best={best_f1:.4f}')
        if wait >= patience:
            if verbose: print(f'  Early stop ep {ep}')
            break
    model.load_state_dict(best_sd)
    return model, best_f1


def eval_test(model, X_te, y_te_, bs=256):
    model.eval()
    dl = DataLoader(TensorDataset(torch.tensor(X_te), torch.tensor(y_te_).long()), batch_size=bs)
    preds, labs = [], []
    with torch.no_grad():
        for X, yy in dl:
            preds.extend(model(X.to(DEVICE)).argmax(1).cpu().numpy())
            labs.extend(yy.numpy())
    return f1_score(labs, preds, average='macro'), np.array(preds), np.array(labs)

print('Training functions ready.')

Training functions ready.


## 10. Pre-train Individual Branches

In [ ]:
print('=== Road-MLP ===')
road_model = RoadMLP(dim=Xr_tr.shape[1])
road_model, road_val_f1 = train_branch(road_model, Xr_tr, Xr_v, y_tr, y_v)
road_test_f1, _, _ = eval_test(road_model, Xr_te, y_te)
print(f'  Road-MLP: val={road_val_f1:.4f} test={road_test_f1:.4f}\n')

print('=== BiLSTM ===')
bilstm_model = BiLSTMBranch(feat=N_FEAT)
bilstm_model, bilstm_val_f1 = train_branch(bilstm_model, Xt_tr, Xt_v, y_tr, y_v)
bilstm_test_f1, _, _ = eval_test(bilstm_model, Xt_te, y_te)
print(f'  BiLSTM: val={bilstm_val_f1:.4f} test={bilstm_test_f1:.4f}\n')

print('=== Tabular-MLP ===')
tab_model = TabularMLP(dim=Xb_tr.shape[1])
tab_model, tab_val_f1 = train_branch(tab_model, Xb_tr, Xb_v, y_tr, y_v)
tab_test_f1, _, _ = eval_test(tab_model, Xb_te, y_te)
print(f'  Tabular-MLP: val={tab_val_f1:.4f} test={tab_test_f1:.4f}\n')

print('--- Unimodal Summary ---')
print(f'{"Branch":<15} {"Val F1":>8} {"Test F1":>8}')
print('-'*33)
for name, vf, tf in [('Road-MLP',road_val_f1,road_test_f1),
                      ('BiLSTM',bilstm_val_f1,bilstm_test_f1),
                      ('Tabular-MLP',tab_val_f1,tab_test_f1)]:
    print(f'{name:<15} {vf:>8.4f} {tf:>8.4f}')

=== Road-MLP ===
  Ep  10 val_f1=0.4089 best=0.4089
  Ep  20 val_f1=0.4227 best=0.4249
  Ep  30 val_f1=0.4124 best=0.4302
  Early stop ep 36
  Road-MLP: val=0.4302 test=0.4343

=== BiLSTM ===
  Ep  10 val_f1=0.3310 best=0.3310
  Ep  20 val_f1=0.3005 best=0.3336
  Early stop ep 23
  BiLSTM: val=0.3336 test=0.3306

=== Tabular-MLP ===
  Ep  10 val_f1=0.4043 best=0.4043
  Ep  20 val_f1=0.3932 best=0.4043
  Early stop ep 22
  Tabular-MLP: val=0.4043 test=0.3927

--- Unimodal Summary ---
Branch            Val F1  Test F1
---------------------------------
Road-MLP          0.4302   0.4343
BiLSTM            0.3336   0.3306
Tabular-MLP       0.4043   0.3927


## 11. Fusion Training (with auxiliary regression)
Multi-task loss: `L = L_focal + 0.3 * L_ordinal + 0.2 * L_regression`

In [ ]:
fusion = CityBrainFusion(road_model, bilstm_model, tab_model).to(DEVICE)
print(f'Fusion params: {sum(p.numel() for p in fusion.parameters()):,}')

tr_dl = DataLoader(FusionDS(
    torch.tensor(Xr_tr), torch.tensor(Xt_tr), torch.tensor(Xb_tr),
    torch.tensor(y_tr).long(), torch.tensor(y_pci_tr)), batch_size=256, shuffle=True)
v_dl = DataLoader(FusionDS(
    torch.tensor(Xr_v), torch.tensor(Xt_v), torch.tensor(Xb_v),
    torch.tensor(y_v).long(), torch.tensor(y_pci_v)), batch_size=256)
te_dl = DataLoader(FusionDS(
    torch.tensor(Xr_te), torch.tensor(Xt_te), torch.tensor(Xb_te),
    torch.tensor(y_te).long(), torch.tensor(y_pci_te)), batch_size=256)

focal = FocalLoss(alpha=CW, gamma=2.0, label_smoothing=0.1)
ordinal = OrdinalPenalty(weight=0.3)
mse = nn.MSELoss()

def eval_fusion(model, dl, disable=None):
    model.eval()
    preds, labs = [], []
    with torch.no_grad():
        for xr,xt,xb,yy,_ in dl:
            xr,xt,xb = xr.to(DEVICE),xt.to(DEVICE),xb.to(DEVICE)
            logits = model.forward_ablation(xr,xt,xb,disable) if disable else model(xr,xt,xb)
            preds.extend(logits.argmax(1).cpu().numpy())
            labs.extend(yy.numpy())
    return f1_score(labs, preds, average='macro'), np.array(preds), np.array(labs)

# Phase 1: Warmup
print('\n--- Phase 1: Warmup (5 ep, branches frozen) ---')
for p in list(fusion.road.parameters()) + list(fusion.bilstm.parameters()) + list(fusion.tabular.parameters()):
    p.requires_grad = False
opt = torch.optim.AdamW(list(fusion.fusion.parameters()) + list(fusion.cls_head.parameters()) + list(fusion.reg_head.parameters()),
                         lr=2e-3, weight_decay=1e-4)
for ep in range(1, 6):
    fusion.train()
    for xr,xt,xb,yy,yr in tr_dl:
        xr,xt,xb,yy,yr = xr.to(DEVICE),xt.to(DEVICE),xb.to(DEVICE),yy.to(DEVICE),yr.to(DEVICE)
        logits, reg_pred = fusion.forward_with_reg(xr,xt,xb)
        loss = focal(logits, yy) + ordinal(logits, yy) + 0.2 * mse(reg_pred, yr)
        opt.zero_grad(); loss.backward(); opt.step()
    f1, _, _ = eval_fusion(fusion, v_dl)
    print(f'  Warmup {ep}/5 val_f1={f1:.4f}')

# Phase 2: End-to-end
print('\n--- Phase 2: End-to-end (60 ep) ---')
for p in fusion.parameters(): p.requires_grad = True
opt = torch.optim.AdamW(fusion.parameters(), lr=2e-4, weight_decay=1e-4)
sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=60)
best_f1, wait, best_sd = 0, 0, None
for ep in range(1, 61):
    fusion.train()
    for xr,xt,xb,yy,yr in tr_dl:
        xr,xt,xb,yy,yr = xr.to(DEVICE),xt.to(DEVICE),xb.to(DEVICE),yy.to(DEVICE),yr.to(DEVICE)
        logits, reg_pred = fusion.forward_with_reg(xr,xt,xb)
        loss = focal(logits, yy) + ordinal(logits, yy) + 0.2 * mse(reg_pred, yr)
        opt.zero_grad(); loss.backward()
        nn.utils.clip_grad_norm_(fusion.parameters(), 1.0); opt.step()
    sch.step()
    f1, _, _ = eval_fusion(fusion, v_dl)
    mk = ''
    if f1 > best_f1: best_f1, wait, best_sd = f1, 0, {k:v.cpu().clone() for k,v in fusion.state_dict().items()}; mk = ' *'
    else: wait += 1
    if ep % 5 == 0: print(f'  Ep {ep:>3} val_f1={f1:.4f} best={best_f1:.4f}{mk}')
    if wait >= 12: print(f'  Early stop ep {ep}'); break

fusion.load_state_dict(best_sd)
fusion = fusion.to(DEVICE)

Fusion params: 208,845

--- Phase 1: Warmup (5 ep, branches frozen) ---
  Warmup 1/5 val_f1=0.4458
  Warmup 2/5 val_f1=0.4490
  Warmup 3/5 val_f1=0.4417
  Warmup 4/5 val_f1=0.4356
  Warmup 5/5 val_f1=0.4331

--- Phase 2: End-to-end (60 ep) ---
  Ep   5 val_f1=0.4383 best=0.4383 *
  Ep  10 val_f1=0.4353 best=0.4383
  Ep  15 val_f1=0.4331 best=0.4383
  Ep  20 val_f1=0.4407 best=0.4407 *
  Ep  25 val_f1=0.4432 best=0.4448
  Ep  30 val_f1=0.4442 best=0.4453
  Ep  35 val_f1=0.4426 best=0.4478
  Ep  40 val_f1=0.4474 best=0.4478
  Early stop ep 44


## 12. Test Evaluation & Ablation

In [ ]:
test_f1, preds, labs = eval_fusion(fusion, te_dl)
print(f'\n{"="*60}')
print(f'FUSION v2 TEST MACRO F1: {test_f1:.4f}  (val best: {best_f1:.4f})')
print(f'{"="*60}')
print(classification_report(labs, preds, target_names=['Low','Medium','High']))

# Ablation
print(f'{"="*60}')
print('ABLATION STUDY')
print(f'{"="*60}')
results = {'Full model': test_f1}
for branch in ['road','bilstm','tabular']:
    ab_f1, _, _ = eval_fusion(fusion, te_dl, disable=branch)
    results[f'w/o {branch}'] = ab_f1

print(f'\n{"Config":<20} {"F1":>6} {"Delta":>8}')
print('-'*36)
for k, v in results.items():
    d = f'{v - test_f1:+.4f}' if k != 'Full model' else '  base'
    print(f'{k:<20} {v:>6.4f} {d:>8}')

print(f'\n--- Fusion v2 vs Unimodal ---')
best_uni = max(road_test_f1, bilstm_test_f1, tab_test_f1)
print(f'Best unimodal: {best_uni:.4f}')
print(f'Fusion v2:     {test_f1:.4f}')
print(f'Improvement:   {test_f1 - best_uni:+.4f}')


FUSION v2 TEST MACRO F1: 0.4358  (val best: 0.4478)
              precision    recall  f1-score   support

         Low       0.57      0.55      0.56       912
      Medium       0.27      0.29      0.28       460
        High       0.46      0.47      0.47       693

    accuracy                           0.46      2065
   macro avg       0.44      0.44      0.44      2065
weighted avg       0.47      0.46      0.47      2065

ABLATION STUDY

Config                   F1    Delta
------------------------------------
Full model           0.4358     base
w/o road             0.3406  -0.0953
w/o bilstm           0.4312  -0.0046
w/o tabular          0.3825  -0.0534

--- Fusion v2 vs Unimodal ---
Best unimodal: 0.4343
Fusion v2:     0.4358
Improvement:   +0.0016


## 13. XGBoost Baseline + Final Comparison

In [ ]:
try:
    from xgboost import XGBClassifier
except ImportError:
    !pip install -q xgboost
    from xgboost import XGBClassifier

X_flat_tr = np.concatenate([Xr_tr, Xt_tr.reshape(len(Xr_tr),-1), Xb_tr], axis=1)
X_flat_v  = np.concatenate([Xr_v,  Xt_v.reshape(len(Xr_v),-1),  Xb_v],  axis=1)
X_flat_te = np.concatenate([Xr_te, Xt_te.reshape(len(Xr_te),-1), Xb_te], axis=1)

xgb = XGBClassifier(n_estimators=500, max_depth=7, learning_rate=0.05,
                     subsample=0.8, colsample_bytree=0.8, min_child_weight=3,
                     eval_metric='mlogloss', random_state=42, verbosity=0)
xgb.fit(X_flat_tr, y_tr, eval_set=[(X_flat_v, y_v)], verbose=False)
xgb_preds = xgb.predict(X_flat_te)
xgb_f1 = f1_score(y_te, xgb_preds, average='macro')
print(f'XGBoost v2 Test Macro F1: {xgb_f1:.4f}')
print(classification_report(y_te, xgb_preds, target_names=['Low','Medium','High']))

# Ensemble: avg logits from fusion + XGBoost proba
xgb_proba = xgb.predict_proba(X_flat_te)
fusion.eval()
fusion_proba = []
with torch.no_grad():
    for xr,xt,xb,yy,_ in te_dl:
        logits = fusion(xr.to(DEVICE),xt.to(DEVICE),xb.to(DEVICE))
        fusion_proba.append(torch.softmax(logits, dim=1).cpu().numpy())
fusion_proba = np.vstack(fusion_proba)

# Weighted ensemble
for w in [0.3, 0.4, 0.5, 0.6, 0.7]:
    ens_proba = w * fusion_proba + (1 - w) * xgb_proba
    ens_preds = ens_proba.argmax(1)
    ens_f1 = f1_score(y_te, ens_preds, average='macro')
    print(f'Ensemble (fusion={w:.1f}): F1={ens_f1:.4f}')

# Best ensemble
best_w = max([(w, f1_score(y_te, (w*fusion_proba + (1-w)*xgb_proba).argmax(1), average='macro'))
              for w in np.arange(0.1, 0.9, 0.05)], key=lambda x: x[1])
ens_best = (best_w[0]*fusion_proba + (1-best_w[0])*xgb_proba).argmax(1)
ens_best_f1 = f1_score(y_te, ens_best, average='macro')
print(f'\nBest ensemble weight: fusion={best_w[0]:.2f}, F1={ens_best_f1:.4f}')
print(classification_report(y_te, ens_best, target_names=['Low','Medium','High']))

print(f'\n{"="*60}')
print('FINAL COMPARISON')
print(f'{"="*60}')
print(f'{"Model":<30} {"Test F1":>8}')
print('-'*40)
print(f'{"v1 Baseline (from prev run)":<30} {"0.3915":>8}')
print(f'{"Road-MLP v2":<30} {road_test_f1:>8.4f}')
print(f'{"BiLSTM v2":<30} {bilstm_test_f1:>8.4f}')
print(f'{"Tabular-MLP v2":<30} {tab_test_f1:>8.4f}')
print(f'{"Fusion v2":<30} {test_f1:>8.4f}')
print(f'{"XGBoost v2":<30} {xgb_f1:>8.4f}')
print(f'{"Ensemble (Fusion+XGB)":<30} {ens_best_f1:>8.4f}')

XGBoost v2 Test Macro F1: 0.4411
              precision    recall  f1-score   support

         Low       0.54      0.73      0.62       912
      Medium       0.35      0.13      0.19       460
        High       0.52      0.50      0.51       693

    accuracy                           0.52      2065
   macro avg       0.47      0.45      0.44      2065
weighted avg       0.49      0.52      0.49      2065

Ensemble (fusion=0.3): F1=0.4455
Ensemble (fusion=0.4): F1=0.4480
Ensemble (fusion=0.5): F1=0.4517
Ensemble (fusion=0.6): F1=0.4533
Ensemble (fusion=0.7): F1=0.4526

Best ensemble weight: fusion=0.80, F1=0.4558
              precision    recall  f1-score   support

         Low       0.57      0.66      0.61       912
      Medium       0.34      0.22      0.26       460
        High       0.49      0.51      0.50       693

    accuracy                           0.51      2065
   macro avg       0.46      0.46      0.46      2065
weighted avg       0.49      0.51      0.49      

---
**End of enhanced pipeline.**

### Summary of improvements over v1:
| Change | Impact |
|--------|--------|
| 311 filtered to pavement types | Cleaner temporal signal |
| +2 temporal channels (pothole, repair) | Type-specific complaint info |
| +dist_to_traffic_count | Proxy for traffic load |
| +near_construction | Construction activity nearby |
| +neigh_mean_pci, +pothole_total | Richer tabular features |
| Focal Loss + Label Smoothing | Better Medium class recall |
| Ordinal penalty | Respect class ordering |
| Auxiliary regression (PCI score) | Multi-task regularization |
| Fusion + XGBoost ensemble | Combine strengths |

# CityBrain v2: Post-Run Deep Analysis & Diagnostics

## 1. The Core Bottleneck: Information Theory vs. Model Architecture
[cite_start]The **Fusion v2 Test Macro F1 (0.4358)** indicates we have hit a **Mutual Information Ceiling** with the current feature set[cite: 755, 775].
* [cite_start]**Evidence 1**: The XGBoost baseline (0.4411) outperforms the complex Fusion model, suggesting the bottleneck is the **feature quality**, not the neural architecture[cite: 839, 877].
* [cite_start]**Evidence 2**: The **BiLSTM branch (0.3306)** performs near-random, providing almost zero incremental value to the fusion[cite: 657, 667, 868].
* [cite_start]**Evidence 3**: **Medium Risk (Label 1)** is the primary failure point ($F1=0.28$), as it represents a transitional state that is easily confused with Low or High boundaries[cite: 787, 885].

---

## 2. Critical Diagnostic: The "Global Consistency" Trap
[cite_start]Our current temporal features (Weather) are **spatially invariant**[cite: 211, 324, 329].
* [cite_start]**The Problem**: Every road segment in the dataset ($N=13,764$) "sees" the exact same weather sequence[cite: 65, 312, 321].
* [cite_start]**The Result**: The BiLSTM cannot learn segment-specific degradation patterns because the input only represents Vancouver's general climate, not the **localized impact** of that climate on specific infrastructure[cite: 506, 519, 657].

---

## 3. High-Priority Iteration Strategy (v3 Roadmap)

### 🔴 P0: Spatial-Temporal Interaction Features
To break the global consistency trap, we must create **interaction terms** that localize weather impacts:
* [cite_start]**Precipitation × Slope**: Higher slopes lead to increased runoff and erosion during heavy rain[cite: 95, 128, 212].
* [cite_start]**Freeze-Thaw × Street Use**: Arterial roads endure higher mechanical stress during freeze-thaw cycles compared to residential streets[cite: 160, 211, 392].
* [cite_start]**Cumulative Stress**: Transition from monthly averages to **cumulative seasonal stress** (e.g., total freeze-thaw cycles since the last survey)[cite: 211, 212].

### 🔴 P0: Architectural Rebalancing
* [cite_start]**Replace BiLSTM with 1D-CNN**: 1D-Convolutional layers are more efficient at capturing "extreme weather window" patterns in short (12-step) sequences[cite: 314, 506].
* [cite_start]**Feature Gating (Attention Fusion)**: Replace raw concatenation with a **Gated Multimodal Fusion** layer to automatically down-weight noisy branches (like the current BiLSTM)[cite: 533, 545, 815].

### 🟡 P1: Tackling the "Medium" Class
* [cite_start]**CORN (Conditional Ordinal Regression)**: Transition from standard Cross-Entropy to a ranking-based loss to respect the ordinal nature of $Low < Medium < High$[cite: 10, 37, 450, 478].
* [cite_start]**Feature Proxy**: Incorporate **Traffic Volume (AADT)** data to better distinguish between "naturally aging" roads and "structurally fatigued" roads[cite: 133, 137, 400].

---

## 4. Summary Table: Expected Improvements
| Priority | Change | Expected $\Delta F1$ | Effort |
| :--- | :--- | :--- | :--- |
| **P0** | Weather × Road Attribute Interactions | $+3\% \sim 5\%$ | Low |
| **P0** | Branch Downsizing (Reduce Overfitting) | $+2\% \sim 3\%$ | Low |
| **P1** | Attention-weighted Fusion Head | $+2\% \sim 3\%$ | Medium |
| **P1** | Ordinal Regression (CORN Loss) | $+2\% \sim 4\%$ | Medium |

---
